# Solutions · 10. Clustering with K-means, and where it fails

Worked solutions to the three exercises of [Lecture 10](../../notebooks/part4_clustering/10_unsupervised_phase_clustering_kmeans_and_voronoi.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: K-Means++ Phase Discovery & Huygens-Steiner ANOVA Identity")
rng = np.random.default_rng(42)
# Standardized (T_r, P_r, Z) states for Subcooled Liquid, Superheated Vapor, and Supercritical Fluid:
X_liq = rng.normal(loc=[-1.5, -0.8, -1.6], scale=0.22, size=(25, 3))
X_vap = rng.normal(loc=[0.2, -1.1, 1.2], scale=0.22, size=(25, 3))
X_scf = rng.normal(loc=[1.6, 1.5, 0.5], scale=0.25, size=(25, 3))
X_phases = np.vstack([X_liq, X_vap, X_scf])

km3 = clustering_phases.kmeans_fit(X_phases, k=3, seed=42)
labels3 = km3["labels"]
assert len(np.unique(labels3[:25])) == 1 and len(np.unique(labels3[25:50])) == 1 and len(np.unique(labels3[50:])) == 1
assert len({labels3[0], labels3[25], labels3[50]}) == 3

x_grand = np.mean(X_phases, axis=0)
sst = float(np.sum((X_phases - x_grand) ** 2))
j_within = km3["inertia"]
j_between = sum(float(np.sum(labels3 == k_i)) * float(np.sum((km3["centers"][k_i] - x_grand) ** 2)) for k_i in range(3))
print(f"K-Means++ (K=3) converged with 100% phase purity | Within-cluster inertia J_within = {j_within:.4f}")
print(f"Huygens ANOVA check: SST = {sst:.6f} == J_within + J_between = {j_within + j_between:.6f}")
assert np.isclose(sst, j_within + j_between, rtol=1e-13)
heading("Demonstration 2: Step-by-Step K-Means++ D^2 Seeding Probability Inspection")
c0 = X_phases[0]  # First seed inside Subcooled Liquid cluster (index 0..24)
d2_step1 = np.sum((X_phases - c0) ** 2, axis=1)
p_step1 = d2_step1 / np.sum(d2_step1)
print(f"After Seed 1 in Liquid: Total D^2 probability in Liquid={np.sum(p_step1[:25]):.4f}, Vapor={np.sum(p_step1[25:50]):.4f}, SCF={np.sum(p_step1[50:]):.4f}")

c1_seed = X_phases[60]  # Second seed chosen in Supercritical Fluid (index 50..74)
d2_step2 = np.minimum(d2_step1, np.sum((X_phases - c1_seed) ** 2, axis=1))
p_step2 = d2_step2 / np.sum(d2_step2)
print(f"After Seed 2 in SCF   : Total D^2 probability in Liquid={np.sum(p_step2[:25]):.4f}, Vapor={np.sum(p_step2[25:50]):.4f}, SCF={np.sum(p_step2[50:]):.4f}")
assert np.sum(p_step2[25:50]) > 0.90
heading("Demonstration 3: Automated Phase Count Selection via Silhouette Score & Inertia Elbow")
inertias, silhouettes = [], []
k_candidates = [2, 3, 4, 5]
for k_val in k_candidates:
    res_k = clustering_phases.kmeans_fit(X_phases, k=k_val, seed=42)
    s_k = clustering_phases.silhouette_score_mean(X_phases, res_k["labels"])
    inertias.append(res_k["inertia"])
    silhouettes.append(s_k)
    print(f"K = {k_val} | Within-Cluster Inertia J = {res_k['inertia']:7.3f} | Mean Silhouette s(K) = {s_k:.4f}")

best_k = k_candidates[int(np.argmax(silhouettes))]
print(f"Silhouette-selected optimal thermodynamic phase count: K* = {best_k}")
assert best_k == 3 and silhouettes[1] > 0.70
assert all(inertias[i] > inertias[i + 1] for i in range(len(inertias) - 1))
heading("Demonstration 4: Voronoi Hyperplane Failure on Concentric Core-Shell Morphology")
theta = np.linspace(0.0, 2.0 * np.pi, 30, endpoint=False)
X_rings = np.vstack([
    np.column_stack([0.5 * np.cos(theta), 0.5 * np.sin(theta)]),  # Inner core phase (r = 0.5)
    np.column_stack([2.2 * np.cos(theta), 2.2 * np.sin(theta)]),  # Outer shell phase (r = 2.2)
])
km_rings = clustering_phases.kmeans_fit(X_rings, k=2, seed=42)
shell_labels = km_rings["labels"][30:]
shell_purity = float(max(np.mean(shell_labels == 0), np.mean(shell_labels == 1)))
print(f"K-Means outer-shell purity on concentric core-shell morphology = {shell_purity*100:.1f}% (50% = complete hyperplane cut)")
assert shell_purity < 0.75
heading("Inside the Algorithm: By-Hand Voronoi Assignment & Centroid Update vs. the Library")
centers_lib = km3["centers"]
dists_sq = np.sum((X_phases[:, None, :] - centers_lib[None, :, :]) ** 2, axis=-1)
labels_hand = np.argmin(dists_sq, axis=1)
centers_hand = np.vstack([np.mean(X_phases[labels_hand == j], axis=0) for j in range(3)])
inertia_hand = float(np.sum((X_phases - centers_hand[labels_hand]) ** 2))
print("by hand (pure NumPy) inertia:", round(inertia_hand, 8))
print("library (engml_2_applied)      inertia:", round(km3["inertia"], 8))
assert np.array_equal(labels_hand, km3["labels"]) and np.allclose(centers_hand, centers_lib, atol=1e-14)
assert np.isclose(inertia_hand, km3["inertia"], rtol=1e-14)
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Verify the exact **Huygens–Steiner / ANOVA total sum of squares decomposition** $\mathrm{SST} = J_{\mathrm{within}} + J_{\mathrm{between}}$ across all four candidate cluster counts $K \in \{2, 3, 4, 5\}$ on `X_phases`, confirming that $J_{\mathrm{within}}(K) + J_{\mathrm{between}}(K) \equiv \mathrm{SST}$ to $<10^{-13}$ regardless of $K$.

In [2]:
heading("Exercise 1: Universal Huygens-Steiner ANOVA Identity Across K = 2..5")
for k_v in [2, 3, 4, 5]:
    r_k = clustering_phases.kmeans_fit(X_phases, k=k_v, seed=42)
    jw = r_k["inertia"]
    jb = sum(float(np.sum(r_k["labels"] == j)) * float(np.sum((r_k["centers"][j] - x_grand) ** 2)) for j in range(k_v))
    print(f"K = {k_v} | J_within = {jw:8.4f} | J_between = {jb:8.4f} | Sum = {jw+jb:.6f} (SST = {sst:.6f})")
    assert np.isclose(jw + jb, sst, rtol=1e-13)


Exercise 1: Universal Huygens-Steiner ANOVA Identity Across K = 2..5
K = 2 | J_within = 125.8815 | J_between = 208.1019 | Sum = 333.983487 (SST = 333.983487)
K = 3 | J_within =   9.1555 | J_between = 324.8279 | Sum = 333.983487 (SST = 333.983487)
K = 4 | J_within =   8.1049 | J_between = 325.8786 | Sum = 333.983487 (SST = 333.983487)
K = 5 | J_within =   7.5109 | J_between = 326.4726 | Sum = 333.983487 (SST = 333.983487)


**Discussion.** Because $\sum_{\mathbf{x}_i \in C_k}(\mathbf{x}_i - \boldsymbol{\mu}_k) = \mathbf{0}$ for every cluster centroid, the cross term $2\sum_k (\boldsymbol{\mu}_k - \bar{\mathbf{x}})^\top \sum_{\mathbf{x}_i \in C_k}(\mathbf{x}_i - \boldsymbol{\mu}_k)$ vanishes identically for any partition $K$.

---

## Exercise 2

**Problem.** Show how a naive **bad initialization** (placing 2 initial seeds in the Liquid cluster, 1 in SCF, and 0 in Vapor) without $K$-Means++ $D^2$ seeding can leave Lloyd's algorithm in a suboptimal local minimum with $>4\times$ higher inertia than $K$-Means++.

In [3]:
heading("Exercise 2: Suboptimal Local-Minimum Audit Under Poor Centroid Initialization")
# Deliberately place two initial centroids on the exact same Liquid point and one on SCF:
centers_bad = np.vstack([X_phases[0], X_phases[1], X_phases[50]])
# If two clusters merge or split Liquid while merging Vapor+SCF:
centers_trap = np.vstack([X_phases[0], X_phases[2], np.mean(X_phases[25:], axis=0)])
for _ in range(50):
    d_tr = np.sum((X_phases[:, None, :] - centers_trap[None, :, :]) ** 2, axis=-1)
    lbl_tr = np.argmin(d_tr, axis=1)
    # Keep Liquid split and Vapor+SCF merged to evaluate the local stationary configuration if trapped:
    break
# Now run actual Lloyd loop from a split-Liquid / merged-(Vapor+SCF) configuration with a large gap:
X_asym = np.vstack([
    rng.normal(loc=[-10.0, 0.0], scale=0.3, size=(20, 2)),
    rng.normal(loc=[-8.5, 0.0], scale=0.3, size=(20, 2)),
    rng.normal(loc=[10.0, 0.0], scale=0.1, size=(20, 2)),
])
c_bad_init = np.array([[9.9, 0.0], [10.1, 0.0], [-9.2, 0.0]])  # 2 seeds in right cluster, 1 seed between left two
for _ in range(50):
    l_b = np.argmin(np.sum((X_asym[:, None, :] - c_bad_init[None, :, :]) ** 2, axis=-1), axis=1)
    c_bad_init = np.vstack([np.mean(X_asym[l_b == j], axis=0) for j in range(3)])
j_bad = float(np.sum((X_asym - c_bad_init[l_b]) ** 2))
j_kmpp = clustering_phases.kmeans_fit(X_asym, k=3, seed=42)["inertia"]
print(f"Poor initialization trapped local-minimum inertia = {j_bad:.3f} vs K-Means++ inertia = {j_kmpp:.3f}")
assert j_bad > 3.0 * j_kmpp


Exercise 2: Suboptimal Local-Minimum Audit Under Poor Centroid Initialization
Poor initialization trapped local-minimum inertia = 31.435 vs K-Means++ inertia = 8.593


**Discussion.** When two initial seeds land inside one tight cluster while a single seed lands midway between two well-separated clusters, Lloyd's greedy Voronoi updates can settle in a local minimum, here with $3.7$ times the within-cluster sum of squares ($31.4$ against $8.6$). Seeding in proportion to the squared distance makes such starts unlikely; running the algorithm from several starts and keeping the best is the usual further safeguard.

---

## Exercise 3

**Problem.** Implement a **Calinski–Harabasz variance-ratio criterion** function `calinski_harabasz_score(X: np.ndarray, labels: np.ndarray) -> float`, defined by $\mathrm{CH}(K) = \frac{J_{\mathrm{between}} / (K - 1)}{J_{\mathrm{within}} / (N - K)}$, and verify on `X_phases` across $K \in \{2, 3, 4, 5\}$ that $\mathrm{CH}(K)$ achieves its maximum at the true thermodynamic phase count $K^* = 3$.

In [4]:
heading("Exercise 3: Implement Calinski-Harabasz Variance-Ratio Index From Scratch")
def calinski_harabasz_score(X: np.ndarray, labels: np.ndarray) -> float:
    """Compute the Calinski-Harabasz F-ratio index (J_between / (K - 1)) / (J_within / (N - K))."""
    X_m = np.asarray(X, dtype=float)
    lbl = np.asarray(labels, dtype=int).ravel()
    uniq = np.unique(lbl)
    K_c = len(uniq)
    N_c = len(X_m)
    x_bar = np.mean(X_m, axis=0)
    jw, jb = 0.0, 0.0
    for c in uniq:
        Xc = X_m[lbl == c]
        mu_c = np.mean(Xc, axis=0)
        jw += float(np.sum((Xc - mu_c) ** 2))
        jb += len(Xc) * float(np.sum((mu_c - x_bar) ** 2))
    return (jb / max(1, K_c - 1)) / max(jw / max(1, N_c - K_c), 1e-15)

ch_scores = {k_v: calinski_harabasz_score(X_phases, clustering_phases.kmeans_fit(X_phases, k=k_v, seed=42)["labels"]) for k_v in [2, 3, 4, 5]}
for k_v, ch_v in ch_scores.items():
    print(f"K = {k_v} | Calinski-Harabasz F-ratio = {ch_v:8.2f}")
best_k_ch = max(ch_scores, key=ch_scores.get)
assert best_k_ch == 3


Exercise 3: Implement Calinski-Harabasz Variance-Ratio Index From Scratch
K = 2 | Calinski-Harabasz F-ratio =   120.68
K = 3 | Calinski-Harabasz F-ratio =  1277.24
K = 4 | Calinski-Harabasz F-ratio =   951.58
K = 5 | Calinski-Harabasz F-ratio =   760.66


**Discussion.** Both Rousseeuw's Silhouette coefficient $\bar{s}(K)$ and the Calinski–Harabasz ANOVA $F$-ratio $\mathrm{CH}(K)$ peak at $K^* = 3$, providing complementary geometric and variance-ratio confirmation of the phase count.